In [ ]:
# Check
import os
import pandas as pd
import nltk
imdb_data=pd.read_csv('IMDB Dataset.csv')
print(imdb_data.shape)
imdb_data.head()

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab') 

In [ ]:
# data collection

import pandas as pd
import re
import os
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.parsing.preprocessing import remove_stopwords
imdb_data=pd.read_csv('IMDB Dataset.csv')

# data cleaning & preprocessing
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize


imdb_data = pd.read_csv('IMDB Dataset.csv')

imdb_data = imdb_data.apply(lambda x: x.str.lower() if x.dtype == 'object' else x)

def clean_text(text):
    text = re.sub(r'<.*?>', '', text)                
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)  
    return text

imdb_data = imdb_data.apply(lambda x: x.apply(clean_text) if x.dtype == 'object' else x)
imdb = imdb_data.apply(lambda x:x.str.lower() if x.dtype == 'object' else x)

# remove stopword
stop_words = set(stopwords.words('english'))
def remove_stopwords_all(text):
    if isinstance(text, str):
        words = text.split()  
        words = [word for word in words if word not in stop_words]
        return ' '.join(words)
    return text

for col in imdb_data.columns:
    if imdb_data[col].dtype == 'object':
        imdb_data[col] = imdb_data[col].apply(remove_stopwords_all)
imdb_data.head()

In [ ]:
# check if positive/negative values equal or not
sentiment_counts = imdb_data['sentiment'].value_counts()
print(sentiment_counts)

In [ ]:
# TF-IDF vectorizer create karo
tfidf_vectorizer = TfidfVectorizer(max_features=1000)

# Reviews column se TF-IDF features banao
tfidf_matrix = tfidf_vectorizer.fit_transform(imdb_data['review'])

# TF-IDF ko dataframe mein convert karo
tfidf_df = pd.DataFrame( tfidf_matrix.toarray(), columns=tfidf_vectorizer.get_feature_names_out()
)

# Original data ke saath TF-IDF features ko merge karo
imdb_data_with_tfidf = pd.concat([imdb_data, tfidf_df], axis=1)

# Result dekho
print("TF-IDF features added successfully!")
print(f"\nOriginal columns: {imdb_data.columns.tolist()}")
print(f"\nNew TF-IDF features (sample 10): {tfidf_df.columns[:10].tolist()}")
print(f"\nFinal dataset shape: {imdb_data_with_tfidf.shape}")
print("\nFirst 5 rows with TF-IDF features:")
imdb_data_with_tfidf.head()

In [ ]:
tfidf smjhao yaha se # Step 1: Import libraries
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

imdb_data = pd.read_csv('IMDB Dataset.csv')

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

imdb_data['review'] = imdb_data['review'].apply(clean_text)

from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

def remove_stopwords_all(text):
    if isinstance(text, str):
        words = text.split()
        words = [word for word in words if word not in stop_words]
        return ' '.join(words)
    return text

imdb_data['review'] = imdb_data['review'].apply(remove_stopwords_all)

# Step 5: TF-IDF vectorizer create karo
print("="*50)
print("TF-IDF Feature Extraction")
print("="*50)

tfidf_vectorizer = TfidfVectorizer(max_features=1000)

# Reviews column se TF-IDF features banao
tfidf_matrix = tfidf_vectorizer.fit_transform(imdb_data['review'])

# TF-IDF ko dataframe mein convert karo
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(), 
    columns=tfidf_vectorizer.get_feature_names_out()
)

# Original data ke saath TF-IDF features ko merge karo
imdb_data_with_tfidf = pd.concat([imdb_data, tfidf_df], axis=1)

# Result dekho
print("TF-IDF features added successfully!")
print(f"\nOriginal columns: {imdb_data.columns.tolist()}")
print(f"\nNew TF-IDF features (sample 10): {tfidf_df.columns[:10].tolist()}")
print(f"\nFinal dataset shape: {imdb_data_with_tfidf.shape}")
print("\nFirst 5 rows with TF-IDF features:")
print(imdb_data_with_tfidf.head())

# Step 6: Model Selection & Training
print("\n" + "="*50)
print("Model Selection & Training")
print("="*50)

# Prepare data for training
X = tfidf_matrix  # TF-IDF features
y = imdb_data['sentiment'].map({'negative': 0, 'positive': 1})  # Labels convert

# Split data into train and test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training data size: {X_train.shape[0]}")
print(f"Testing data size: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]}")

# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Naive Bayes': MultinomialNB(),
    'SVM': SVC(kernel='linear')
}

print("\n" + "="*50)
print("Model Training Started")
print("="*50)

trained_models = {}

# Train each model

for model_name, model in models.items():
    print(f"\nTraining {model_name}...")
    model.fit(X_train, y_train)
    trained_models[model_name] = model
    print(f"{model_name} trained successfully!")

print("\n" + "="*50)
print("Model Evaluation Results")
print("="*50)

for model_name, model in trained_models.items():
    y_pred = model.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    
    print(f"\n{'='*30}")
    print(f"{model_name}")
    print(f"{'='*30}")
    print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))
    
    cm = confusion_matrix(y_test, y_pred)
    print(f"\nConfusion Matrix:")
    print(f"                 Predicted")
    print(f"                 Neg   Pos")
    print(f"Actual  Neg     {cm[0,0]:5d} {cm[0,1]:5d}")
    print(f"        Pos     {cm[1,0]:5d} {cm[1,1]:5d}")
    
    # Additional insights
    tn, fp, fn, tp = cm.ravel()
    print(f"\nDetailed Analysis:")
    print(f" True Negatives (Correct Negative): {tn}")
    print(f" False Positives (Wrong Positive): {fp}")
    print(f" False Negatives (Wrong Negative): {fn}")
    print(f" True Positives (Correct Positive): {tp}")

# Step 8: Confusion Matrix Visualization
print("\n" + "="*50)
print("Confusion Matrix Visualization")
print("="*50)

best_model = trained_models['Logistic Regression']
y_pred_best = best_model.predict(X_test)
cm_best = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_best, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix - Logistic Regression (Best Model)')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()

# Step 9: Model Performance Summary
print("\n" + "="*50)
print("Model Performance Summary")
print("="*50)

for model_name, model in trained_models.items():
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    print(f"\n{model_name}:")
    print(f"  • Accuracy: {accuracy*100:.2f}%")
    print(f"  • False Positives (Negative ko Positive bola): {fp}")
    print(f"  • False Negatives (Positive ko Negative bola): {fn}")
    
    # Precision = TP / (TP + FP)
    if (tp+fp) > 0:
        print(f"  • Precision: {tp/(tp+fp):.3f}")
    else:
        print(f"  • Precision: N/A")
    
    # Recall = TP / (TP + FN)
    if (tp+fn) > 0:
        print(f"  • Recall: {tp/(tp+fn):.3f}")
    else:
        print(f"  • Recall: N/A")

# Step 10: Real-time Sentiment Predictor
print("\n" + "="*50)
print("Real-time Sentiment Predictor")
print("="*50)

def predict_sentiment(review_text, model=trained_models['Logistic Regression']):
    # Clean the review
    review_text = review_text.lower()
    review_text = clean_text(review_text)
    review_text = remove_stopwords_all(review_text)
    
    review_tfidf = tfidf_vectorizer.transform([review_text])
    
    # Predict
    prediction = model.predict(review_tfidf)[0]
    probability = model.predict_proba(review_tfidf)[0] if hasattr(model, 'predict_proba') else None
    
    sentiment = "Positive" if prediction == 1 else "Negative"
    
    print(f"\nReview: {review_text[:100]}...")
    print(f"Predicted Sentiment: {sentiment}")
    if probability is not None:
        print(f"Confidence: {max(probability)*100:.2f}%")
    
    return sentiment

print("\nTesting on sample reviews:")
print("-"*40)
predict_sentiment("This movie was amazing! I loved it completely.")
print("-"*40)
predict_sentiment("Worst movie ever. Complete waste of time.")
print("-"*40)

print("\n" + "="*50)
print("Test Your Own Review")
print("="*50)
user_review = input("Enter your movie review: ")
if user_review.strip():
    predict_sentiment(user_review)

TF-IDF Feature Extraction
TF-IDF features added successfully!

Original columns: ['review', 'sentiment']

New TF-IDF features (sample 10): ['able', 'absolutely', 'across', 'act', 'acted', 'acting', 'action', 'actor', 'actors', 'actress']

Final dataset shape: (50000, 1002)

First 5 rows with TF-IDF features:
                                              review sentiment  able  \
0  one reviewers mentioned watching oz episode yo...  positive   0.0   
1  wonderful little production br br filming tech...  positive   0.0   
2  thought wonderful way spend time hot summer we...  positive   0.0   
3  basically theres family little boy jake thinks...  negative   0.0   
4  petter matteis love time money visually stunni...  positive   0.0   

   absolutely  across  act  acted   acting    action  actor  ...     years  \
0         0.0     0.0  0.0    0.0  0.00000  0.000000    0.0  ...  0.000000   
1         0.0     0.0  0.0    0.0  0.00000  0.000000    0.0  ...  0.000000   
2         0.0     0.0  

In [ ]:
user_review = input("Enter your movie review: ")
if user_review.strip():
    predict_sentiment(user_review)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Sentiment count check karna
sns.countplot(x='sentiment', data=imdb_data)
plt.title('Positive vs Negative Reviews')
plt.show()

imdb_data['length'] = imdb_data['sentiment'].apply(len)
imdb_data['length'].hist(bins=50)
plt.title('Review Length Distribution')
plt.show()